In [35]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [36]:
class chatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm= init_chat_model("google_genai:gemini-2.5-flash")
def chat_node(state: chatState):
    messages = state['messages']  
    response = llm.invoke(messages)
    return {'messages':[response]}

In [38]:
checkpointr = MemorySaver()
graph = StateGraph(chatState)
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)


chatbot = graph.compile(checkpointer=checkpointr)

In [40]:
# initial_state = {
#     'messages': [SystemMessage(content="You are a helpful assistant."),
#                  HumanMessage(content="What is the capital of India?")]
# }

# response=chatbot.invoke(initial_state)['messages']


In [41]:
# print(response)

In [42]:
thread_id = '1'



while True:
    user_message = input("User: ")
    print("User:", user_message)
    if user_message.lower().strip() in ['exit', 'quit']:
        print("Exiting chatbot. Goodbye!")
        break

    config = {
        'configurable': {
            'thread_id': thread_id
    }
    }


    response  = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print("Chatbot:", response['messages'][-1].content)
    

User: what is the capital of bhutan 
Chatbot: The capital of Bhutan is **Thimphu**.
User: hey my name is sarthak 
Chatbot: Hi Sarthak, it's nice to meet you! How can I help you today?
User: what is my name 
Chatbot: Your name is **Sarthak**.
User: my date of birth is 8th jan 2003
Chatbot: Thank you for sharing that, Sarthak. So, your date of birth is January 8th, 2003. Got it!
User: what i my date of birth 
Chatbot: Your date of birth is **January 8th, 2003**.
User: exit 
Exiting chatbot. Goodbye!


In [44]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='what is the capital of bhutan ', additional_kwargs={}, response_metadata={}, id='01d9a6e6-e6b8-4dd3-9056-fb1bfbe26817'), AIMessage(content='The capital of Bhutan is **Thimphu**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ed07e-c5cf-7191-b001-d7978170518e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 26, 'total_tokens': 35, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 15}}), HumanMessage(content='hey my name is sarthak ', additional_kwargs={}, response_metadata={}, id='8f1d3d52-b5a9-4776-ab0e-aa29144bac2f'), AIMessage(content="Hi Sarthak, it's nice to meet you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provi